# Fine-tune Laya for HTML phishing classification (Kaggle GPU)

Runs the same experiment as `train_html_phishing_classifier.ipynb`, with Laya's
ModernBERT-large decision model in place of MarkupLM. Sampling, splits, seed and
early stopping are copied from that notebook unchanged, so the only variable that
moves is the model.

**Run this on Kaggle with a GPU accelerator and internet enabled.** It will not run on
Apple Silicon: the fine-tune needs CUDA, and `torch.compile` (which ModernBERT uses)
is unsupported on Python 3.14.

What to compare the result against, all measured earlier on these same splits:

| | AUC on held-out PhreshPhish | AUC on Mendeley | Handwritten |
|---|---|---|---|
| Page length alone | 0.739 | 0.745 | — |
| MarkupLM `v0` | 0.751 | — | — |
| **MarkupLM `v1`** (756 pages) | **0.869** | **0.814** | 12/14 |
| Laya zero-shot | not measured | 0.724 | 11/14 |

Zero-shot Laya lost to the length baseline on Mendeley, and its errors were lopsided:
recall 0.873 but precision 0.621, because "is this phishing?" collapsed into "does this
have a login form?". Fine-tuning is the test of whether that is fixable.

### One deliberate deviation

Laya's published checkpoints were trained with RLCD, a reinforcement objective against
proper scoring rules. This notebook trains with cross-entropy instead, because that is
what `train_html_phishing_classifier.ipynb` does and matching it keeps the comparison
honest. Calibration is recovered afterwards by fitting a temperature on the validation
split, which is Laya's own procedure. `laya.common.proper_reward` is available if you
later want to add the RL term.


In [ ]:
!pip install -q -U "laya>=0.3.0" "transformers>=4.48.0" safetensors huggingface_hub pyarrow scikit-learn
import torch, transformers, laya
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU accelerator")

## 1. Configuration

Sampling constants are `train_html_phishing_classifier.ipynb`'s, unchanged, so the split
is the one MarkupLM `v1` trained on. Learning rates come from Laya's own fine-tuning
recipe: the encoder moves slowly, the freshly-shaped head faster.

`MICRO_BATCH` is deliberately small. ModernBERT-large is 421M parameters against
MarkupLM-base's 136M, and a T4 has 16 GB. Raise it if `nvidia-smi` shows headroom; lower
it to 2 if you hit an out-of-memory error.


In [ ]:
import json, math, random, hashlib, re, os, time
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlsplit

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from transformers import AutoTokenizer

SEED = 42
# The sampling cell below is copied verbatim from the MarkupLM notebook and
# expects this mapping, which lives in that notebook's imports cell.
PHRESHPHISH_LABEL_TO_ID = {"benign": 0, "phish": 1}
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

BASE_REPO = "convaiinnovations/laya"          # English checkpoint, ModernBERT-large
ARTIFACT_DIR = Path("laya-phishing-v1")        # written at the end, download from Kaggle output

# Laya's fine-tuning recipe
ENCODER_LR = 2.5e-5
HEAD_LR = 1.0e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
MICRO_BATCH = 4
GRAD_ACCUM = 8                                 # effective batch 32
MAX_LEN = 512                                  # the English checkpoint's context
HEAD_MAX_LEN = 192

# The question the model is being taught to answer. Index 1 is "yes" (laya/agent.py).
QUESTION = {"t": "noul",
            "ins": ("Is this web page a phishing page that impersonates a real brand "
                    "to steal user credentials?"),
            "crit": None}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This notebook needs a GPU. On Kaggle: Settings > Accelerator > GPU.")
# T4 is Turing and has no bfloat16; A100/P100 differ, so ask rather than assume.
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("device:", torch.cuda.get_device_name(0), "| amp:", AMP_DTYPE)

## 2. Sampling — copied verbatim from the MarkupLM notebook

Every line below is `train_html_phishing_classifier.ipynb`'s sampling cell, unchanged.
Same seed, same shards, same short-page and login quotas, same hostname-based split.
Changing anything here would break the comparison it exists to support.


In [ ]:
DATASET_ID = "phreshphish/phreshphish"
CACHE_DIR = Path("datasets/phreshphish_sample/html_v1_short_pages")
USE_TRAIN_PER_CLASS = 400
USE_VAL_PER_CLASS = 100
USE_TEST_PER_CLASS = 100
TRAIN_SHARDS = 4
TEST_SHARDS = 3
MIN_HTML_CHARS = 20
MAX_HTML_CHARS = 1_000_000
PARSE_CHAR_BUDGET = 300_000
SHORT_HTML_CHARS = 2_000
SHORT_FRACTION = 0.20
BENIGN_LOGIN_FRACTION = 0.20
LOGIN_RE = re.compile(r"<input\b[^>]*\btype\s*=\s*['\"]?password\b", re.I)


def normalize_page(row):
    html = row.get("html") or ""
    label = row.get("label")
    if label not in PHRESHPHISH_LABEL_TO_ID or not MIN_HTML_CHARS <= len(html) <= MAX_HTML_CHARS:
        return None
    url = row.get("url") or ""
    try:
        host = (urlsplit(url).hostname or "").lower().rstrip(".")
    except ValueError:
        host = ""
    # Hash before truncation, so different tails are not mistaken for identical pages.
    digest = hashlib.sha256(html.encode("utf-8", "ignore")).hexdigest()
    return {"html": html[:PARSE_CHAR_BUDGET], "html_len": len(html),
            "label": label, "label_id": PHRESHPHISH_LABEL_TO_ID[label], "url": url,
            "host": host, "sha256": digest,
            "is_login": bool(LOGIN_RE.search(html)), "short": len(html) <= SHORT_HTML_CHARS}


def host_group(row):
    return "host:" + row["host"] if row["host"] else "hash:" + row["sha256"]


def reservoir_add(bucket, row, count, capacity, rng):
    if len(bucket) < capacity:
        bucket.append(row)
    else:
        position = rng.randrange(count)
        if position < capacity:
            bucket[position] = row


def select_coverage(uniform, short, login, limit, short_fraction, login_fraction):
    selected, seen = [], set()
    def add(rows, target):
        added = 0
        for row in rows:
            if len(selected) >= limit or added >= target:
                break
            if row["sha256"] not in seen:
                selected.append(row)
                seen.add(row["sha256"])
                added += 1
    add(short, round(limit * short_fraction))
    add(login, round(limit * login_fraction))
    add(uniform, limit)
    return selected


def sample_shard(local_path, per_class, seed, enrich):
    rng = random.Random(seed)
    pools = {(label, kind): [] for label in (0, 1) for kind in ("uniform", "short", "login")}
    counts = Counter()
    for batch in pq.ParquetFile(local_path).iter_batches(batch_size=64, columns=["html", "url", "label"]):
        for raw in batch.to_pylist():
            row = normalize_page(raw)
            if row is None:
                continue
            label = row["label_id"]
            kinds = ["uniform"]
            if enrich and row["short"]:
                kinds.append("short")
            if enrich and label == 0 and row["is_login"]:
                kinds.append("login")
            for kind in kinds:
                key = (label, kind)
                counts[key] += 1
                reservoir_add(pools[key], row, counts[key], per_class, rng)
    selected = []
    for label in (0, 1):
        for kind in ("uniform", "short", "login"):
            rng.shuffle(pools[(label, kind)])
        selected += select_coverage(
            pools[(label, "uniform")], pools[(label, "short")], pools[(label, "login")],
            per_class, SHORT_FRACTION if enrich else 0,
            BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0,
        )
    return selected


def sample_split(split, per_class, shard_count, enrich):
    files = sorted(f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset")
                   if f.startswith(f"data/{split}-") and f.endswith(".parquet"))
    if not files:
        raise RuntimeError(f"No parquet shards for {split}")
    count = min(shard_count, len(files))
    indices = [round(i * (len(files) - 1) / max(count - 1, 1)) for i in range(count)]
    rows = []
    for index in indices:
        shard = files[index]
        local = hf_hub_download(DATASET_ID, shard, repo_type="dataset")
        sampled = sample_shard(local, math.ceil(per_class / count), SEED + index, enrich)
        for row in sampled:
            row["shard"] = shard
        rows.extend(sampled)
        print(shard, "selected:", len(sampled))
    return rows


def deduplicate(rows):
    by_hash, conflicting = {}, set()
    for row in rows:
        key = row["sha256"]
        if key in by_hash and by_hash[key]["label_id"] != row["label_id"]:
            conflicting.add(key)
        by_hash[key] = row
    return [r for key, r in by_hash.items() if key not in conflicting]


def choose_balanced(rows, per_class, seed, enrich=False):
    rng = random.Random(seed)
    result = []
    for label in (0, 1):
        group = [r for r in rows if r["label_id"] == label]
        rng.shuffle(group)
        result += select_coverage(group, [r for r in group if r["short"]],
                                  [r for r in group if r["is_login"] and label == 0], per_class,
                                  SHORT_FRACTION if enrich else 0,
                                  BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0)
    rng.shuffle(result)
    return result


def split_hosts(rows, val_per_class, seed):
    groups = defaultdict(list)
    for row in rows:
        groups[host_group(row)].append(row)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    val, val_groups, counts = [], set(), Counter()
    for key in keys:
        group = groups[key]
        additions = Counter(r["label_id"] for r in group)
        if all(counts[label] + additions[label] <= val_per_class for label in (0, 1)):
            val.extend(group)
            counts.update(additions)
            val_groups.add(key)
    return [r for r in rows if host_group(r) not in val_groups], val


def assert_separate(splits):
    previous_hosts, previous_hashes = set(), set()
    for name, rows in splits.items():
        hosts, hashes = {host_group(r) for r in rows}, {r["sha256"] for r in rows}
        if hosts & previous_hosts or hashes & previous_hashes:
            raise ValueError(f"Host/content overlap in {name}")
        if len(hashes) != len(rows):
            raise ValueError(f"Duplicate content in {name}")
        if set(r["label_id"] for r in rows) != {0, 1}:
            raise ValueError(f"{name} needs both classes; increase shard/sample counts")
        previous_hosts.update(hosts)
        previous_hashes.update(hashes)


sample_config = {"version": 1, "dataset": DATASET_ID, "seed": SEED,
                 "train_shards": TRAIN_SHARDS, "test_shards": TEST_SHARDS,
                 "train_pool_per_class": 2 * (USE_TRAIN_PER_CLASS + USE_VAL_PER_CLASS),
                 "test_pool_per_class": 3 * USE_TEST_PER_CLASS,
                 "min_chars": MIN_HTML_CHARS, "max_chars": MAX_HTML_CHARS,
                 "parse_budget": PARSE_CHAR_BUDGET, "short_chars": SHORT_HTML_CHARS,
                 "short_fraction": SHORT_FRACTION, "login_fraction": BENIGN_LOGIN_FRACTION}
cache_metadata = CACHE_DIR / "sampling.json"
cache_valid = (cache_metadata.exists()
               and json.loads(cache_metadata.read_text(encoding="utf-8")) == sample_config
               and all((CACHE_DIR / f"{name}.jsonl").exists() for name in ("pool", "test")))
if cache_valid:
    def read_cache(name):
        with (CACHE_DIR / f"{name}.jsonl").open(encoding="utf-8") as handle:
            return [json.loads(line) for line in handle]
    pool, test_pool = read_cache("pool"), read_cache("test")
else:
    pool = sample_split("train", sample_config["train_pool_per_class"], TRAIN_SHARDS, True)
    test_pool = sample_split("test", sample_config["test_pool_per_class"], TEST_SHARDS, False)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for name, rows in (("pool", pool), ("test", test_pool)):
        with (CACHE_DIR / f"{name}.jsonl").open("w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    cache_metadata.write_text(json.dumps(sample_config, indent=2), encoding="utf-8")

pool = deduplicate(pool)
train_pool, val_raw = split_hosts(pool, USE_VAL_PER_CLASS, SEED)
train_raw = choose_balanced(train_pool, USE_TRAIN_PER_CLASS, SEED, enrich=True)
known_hosts = {host_group(r) for r in train_raw + val_raw}
known_hashes = {r["sha256"] for r in pool}
test_pool = deduplicate(test_pool)
eligible_test = [r for r in test_pool if host_group(r) not in known_hosts and r["sha256"] not in known_hashes]
print("Excluded test pages with seen hosts/content:", len(test_pool) - len(eligible_test))
test_raw = choose_balanced(eligible_test, USE_TEST_PER_CLASS, SEED + 1)
assert_separate({"train": train_raw, "val": val_raw, "test": test_raw})
for name, rows in (("train", train_raw), ("val", val_raw), ("test", test_raw)):
    print(name, "labels:", dict(Counter(r["label_id"] for r in rows)),
          "short benign:", sum(r["short"] and r["label_id"] == 0 for r in rows),
          "benign login:", sum(r["is_login"] and r["label_id"] == 0 for r in rows))

## 3. HTML preprocessing

The same cleaning MarkupLM `v1` recorded in its metadata — scripts, styles, noscript and
template elements removed, comments stripped, 300k characters at most.

The output differs in one way that matters: MarkupLM consumed `(nodes, xpaths)` pairs and
saw the DOM as structure. Laya reads plain text, so it gets the cleaned markup as a
string. Tags cost tokens, but measured on the handwritten examples, keeping them beat
stripping to visible text (11/14 against 10/14), because `<form>` and `<input type=password>`
are exactly the giveaway.


In [ ]:
from bs4 import BeautifulSoup, Comment

STRIP_TAGS = ["script", "style", "noscript", "template"]


def clean_html(html):
    soup = BeautifulSoup(html[:PARSE_CHAR_BUDGET], "html.parser")
    for element in soup.find_all(STRIP_TAGS):
        element.decompose()
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    text = str(soup)
    if not text.strip():
        raise ValueError("No usable markup")
    return text


parse_failures = []


def build_pages(raw_examples, split):
    pages = []
    for item in raw_examples:
        try:
            pages.append({"text": clean_html(item["html"]), "label_id": item["label_id"],
                          "html_chars": len(item["html"]), "sha256": item["sha256"],
                          "host": item["host"], "short": item["short"], "is_login": item["is_login"]})
        except Exception as exc:
            parse_failures.append({"split": split, "sha256": item["sha256"], "reason": type(exc).__name__})
    return pages


train_pages = build_pages(train_raw, "train")
val_pages = build_pages(val_raw, "val")
test_pages = build_pages(test_raw, "test")
for name, pages in (("train", train_pages), ("val", val_pages), ("test", test_pages)):
    print(name, len(pages), dict(Counter(p["label_id"] for p in pages)))
print("parse failures:", dict(Counter(r["split"] for r in parse_failures)))

## 4. Turning pages into Laya training items

Laya does not take a label directly. Each item is a tokenised sequence built from the
state and the question, plus the marker positions where the two answer options live, and
a target distribution over them.

For a `noul` question the options are ordered `[false, true]`, so a phishing page is
`[0.0, 1.0]` and a benign page `[1.0, 0.0]`. Getting that order backwards would train the
model to answer the exact opposite of what is intended, so it is asserted below rather
than trusted.


In [ ]:
from laya.common import build_sequence, collate_items, QTYPES

tokenizer_dir = snapshot_download(BASE_REPO, allow_patterns=["tokenizer/*"])
tok = AutoTokenizer.from_pretrained(os.path.join(tokenizer_dir, "tokenizer"))
PAD_ID = tok.pad_token_id
NOUL = QTYPES["noul"]

# Index 1 must be "true": laya/agent.py reports noul as p[1].
_probe_ids, _probe_markers = build_sequence(tok, "probe", QUESTION, max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN)
assert len(_probe_markers) == 2, f"expected two options for a noul question, got {len(_probe_markers)}"


def to_item(page):
    ids, markers = build_sequence(tok, page["text"], QUESTION,
                                  max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN)
    label = int(page["label_id"])          # 1 = phishing
    target = [0.0, 1.0] if label == 1 else [1.0, 0.0]
    return {"ids": ids, "markers": markers, "qtype": NOUL, "target": target, "label": label}


train_items = [to_item(p) for p in train_pages]
val_items = [to_item(p) for p in val_pages]
test_items = [to_item(p) for p in test_pages]
print(f"items: train {len(train_items)}  val {len(val_items)}  test {len(test_items)}")

_lengths = [len(it["ids"]) for it in train_items]
_capped = sum(l >= MAX_LEN for l in _lengths)
print(f"token length: median {sorted(_lengths)[len(_lengths)//2]}, "
      f"{_capped}/{len(_lengths)} ({_capped/len(_lengths):.0%}) hit the {MAX_LEN}-token cap")


def batches(items, size, shuffle=False):
    order = list(range(len(items)))
    if shuffle:
        random.Random(SEED).shuffle(order)
    for start in range(0, len(order), size):
        group = [[items[i]] for i in order[start:start + size]]
        yield collate_items(group, PAD_ID)

## 5. Load the checkpoint

`build_model` reconstructs the decision model from `rl_agent_config.json`, then the
published weights are loaded into it. Gradient checkpointing trades recompute for memory,
which is what lets a 421M-parameter encoder train on a 16 GB card.


In [ ]:
from laya.common import build_model
from safetensors.torch import load_file

model_dir = snapshot_download(BASE_REPO, allow_patterns=["model.safetensors", "encoder/*", "rl_agent_config.json"])
cfg = json.loads(Path(model_dir, "rl_agent_config.json").read_text())
print("encoder:", cfg["encoder"], "| head_layers:", cfg["head_layers"], "| shipped temps:", cfg.get("temperature"))

model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"), pretrained=False)
missing, unexpected = model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=False)
if missing or unexpected:
    print("missing:", missing[:5], "| unexpected:", unexpected[:5])
model = model.to(device)
model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

total = sum(p.numel() for p in model.parameters())
print(f"parameters: {total/1e6:.0f}M  (MarkupLM-base is 136M)")

## 6. Train

Cross-entropy over the two option logits, which is the same objective
`train_html_phishing_classifier.ipynb` uses. Selection is also the same: the checkpoint
with the lowest validation loss is restored before anything is tested or saved, and two
epochs without improvement stop the run.

Two parameter groups, following Laya's recipe — the pretrained encoder at 2.5e-5, the
decision head at 1e-4.


In [ ]:
encoder_params, head_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    (encoder_params if name.startswith("encoder.") else head_params).append(param)
print(f"encoder tensors {len(encoder_params)} | head tensors {len(head_params)}")

optimizer = torch.optim.AdamW(
    [{"params": encoder_params, "lr": ENCODER_LR},
     {"params": head_params, "lr": HEAD_LR}], weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_items) / MICRO_BATCH / GRAD_ACCUM)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps_per_epoch * NUM_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
print(f"{steps_per_epoch} optimizer steps/epoch, effective batch {MICRO_BATCH * GRAD_ACCUM}")


def forward_loss(batch):
    batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    logits, _ = model(batch["input_ids"], batch["attention_mask"],
                      batch["marker_pos"], batch["marker_mask"], batch["qtype"])
    logp = torch.log_softmax(logits.masked_fill(~batch["marker_mask"], -1e4), dim=-1)
    loss = -(batch["target"] * logp).sum(-1).mean()
    return loss, logits.detach(), batch["label"]


@torch.no_grad()
def evaluate(items):
    model.eval()
    total_loss, scores, labels = 0.0, [], []
    for batch in batches(items, MICRO_BATCH):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss, logits, label = forward_loss(batch)
        total_loss += loss.item() * len(label)
        scores.extend(torch.softmax(logits.float(), -1)[:, 1].cpu().tolist())
        labels.extend(label.cpu().tolist())
    predictions = [int(s >= 0.5) for s in scores]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", pos_label=1, zero_division=0)
    return ({"loss": total_loss / len(labels), "accuracy": accuracy_score(labels, predictions),
             "precision_phishing": precision, "recall_phishing": recall, "f1_phishing": f1,
             "auc": roc_auc_score(labels, scores), "n": len(labels)}, scores, labels)


history, best_state, best_loss, best_epoch, stale = [], None, float("inf"), None, 0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running, seen, step = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)
    started = time.time()
    for batch in batches(train_items, MICRO_BATCH, shuffle=True):
        step += 1
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss, _, label = forward_loss(batch)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss at epoch {epoch} step {step}")
        scaler.scale(loss / GRAD_ACCUM).backward()
        if step % GRAD_ACCUM == 0 or step * MICRO_BATCH >= len(train_items):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad(set_to_none=True)
        running += loss.item() * len(label); seen += len(label)
    val_metrics, _, _ = evaluate(val_items)
    history.append({"epoch": epoch, "train_loss": running / seen, "val": val_metrics,
                    "seconds": round(time.time() - started)})
    print(json.dumps(history[-1]))
    if val_metrics["loss"] < best_loss:
        best_loss, best_epoch, stale = val_metrics["loss"], epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  best so far (epoch {epoch}, val_loss {best_loss:.4f})")
    else:
        stale += 1
        if stale >= EARLY_STOPPING_PATIENCE:
            print("early stopping: validation loss stopped improving")
            break

if best_state is None:
    raise RuntimeError("no checkpoint was produced")
model.load_state_dict(best_state)
model.eval()
print("restored epoch", best_epoch)

## 7. Calibrate

Laya's selling point is that its probabilities mean something, and cross-entropy training
does not preserve that on its own. A single temperature is fitted on the validation split
with LBFGS, the same way Laya fits its shipped temperatures, and it is written into the
saved config so inference applies it automatically.

The temperature is fitted on validation, never on test — a temperature tuned on the test
split would make the reported calibration meaningless.


In [ ]:
_, val_scores, val_labels = evaluate(val_items)


@torch.no_grad()
def collect_logits(items):
    model.eval()
    out = []
    for batch in batches(items, MICRO_BATCH):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits, _ = model(batch["input_ids"], batch["attention_mask"],
                              batch["marker_pos"], batch["marker_mask"], batch["qtype"])
        out.append(logits.float().cpu())
    return torch.cat(out)


val_logits = collect_logits(val_items)
val_target = torch.tensor([[0.0, 1.0] if it["label"] == 1 else [1.0, 0.0] for it in val_items])

log_t = torch.zeros(1, requires_grad=True)
lbfgs = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)


def closure():
    lbfgs.zero_grad()
    loss = -(val_target * torch.log_softmax(val_logits / log_t.exp(), -1)).sum(-1).mean()
    loss.backward()
    return loss


lbfgs.step(closure)
TEMPERATURE = float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


def nll(logits, target, temperature):
    return float(-(target * torch.log_softmax(logits / temperature, -1)).sum(-1).mean())


print(f"fitted temperature {TEMPERATURE:.4f}")
print(f"  validation NLL  {nll(val_logits, val_target, 1.0):.4f} -> {nll(val_logits, val_target, TEMPERATURE):.4f}")

## 8. Test

The held-out split, scored once, with the calibrated temperature applied. AUC is the
number to carry back: it does not move when the threshold moves, so it compares directly
against MarkupLM `v1`'s 0.869 and the 0.739 length-only floor.

A trivial baseline is recomputed here on the same pages. A fine-tuned 421M model that
cannot beat character counting has not learned the task.


In [ ]:
test_logits = collect_logits(test_items)
test_scores = torch.softmax(test_logits / TEMPERATURE, -1)[:, 1].tolist()
test_labels = [it["label"] for it in test_items]
predictions = [int(s >= 0.5) for s in test_scores]

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, predictions, average="binary", pos_label=1, zero_division=0)
test_metrics = {"accuracy": accuracy_score(test_labels, predictions),
                "precision_phishing": precision, "recall_phishing": recall, "f1_phishing": f1,
                "auc": roc_auc_score(test_labels, test_scores),
                "temperature": TEMPERATURE, "n": len(test_labels)}

length_baseline = roc_auc_score(test_labels, [-p["html_chars"] for p in test_pages])

print(json.dumps(test_metrics, indent=2))
print("\nconfusion [[TN, FP], [FN, TP]]:")
print(confusion_matrix(test_labels, predictions, labels=[0, 1]))
print(f"\n{'Laya fine-tuned':28} AUC {test_metrics['auc']:.3f}")
print(f"{'page length alone':28} AUC {length_baseline:.3f}   <- same pages")
print(f"{'MarkupLM v1':28} AUC 0.869   <- same split, 756 pages")
print(f"{'Laya zero-shot (Mendeley)':28} AUC 0.724")

if test_metrics["auc"] <= length_baseline:
    print("\nFine-tuning did not lift it past character counting. Check the token cap above: "
          "if most pages hit 512, the model is reading headers and nothing else.")
elif test_metrics["auc"] >= 0.869:
    print("\nBeat MarkupLM v1 on its own split.")
else:
    print(f"\nBelow MarkupLM v1 by {0.869 - test_metrics['auc']:.3f} AUC, but clear of the "
          "length baseline, so the fine-tune did teach it something.")

## 9. Save

Written in Laya's own artifact layout so `laya.Agent(path)` can load it directly, plus a
`metadata.json` in the shape `smoke_test.ipynb` expects. On Kaggle the folder lands in
`/kaggle/working`; download it from the Output tab and drop it into
`ai/markuplm/artifacts/` to score it with `smoke_test_laya.ipynb`.


In [ ]:
from safetensors.torch import save_file

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
state = {k: v.detach().half().contiguous().cpu() for k, v in model.state_dict().items()}
save_file(state, str(ARTIFACT_DIR / "model.safetensors"))
model.encoder.config.save_pretrained(str(ARTIFACT_DIR / "encoder"))
tok.save_pretrained(str(ARTIFACT_DIR / "tokenizer"))

out_cfg = dict(cfg)
# temperature is indexed by question type: [choice, score, noul]
temps = list(out_cfg.get("temperature", [1.0, 1.0, 1.0]))
temps[QTYPES["noul"]] = TEMPERATURE
out_cfg["temperature"] = temps
out_cfg["temperature_by_options"] = {**out_cfg.get("temperature_by_options", {}), "noul:2": TEMPERATURE}
out_cfg["model_name"] = "laya-phishing-v1"
(ARTIFACT_DIR / "rl_agent_config.json").write_text(json.dumps(out_cfg, indent=2))

metadata = {
    "base_model": BASE_REPO,
    "task": "sequence-classification",
    "id2label": {0: "benign", 1: "phishing"},
    "train_dataset": "phreshphish/phreshphish",
    "seed": SEED,
    "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
    "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
    "effective_batch_size": MICRO_BATCH * GRAD_ACCUM,
    "max_epochs": NUM_EPOCHS, "best_epoch": best_epoch,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "selection_metric": "lowest validation cross-entropy",
    "history": history,
    "train_n": len(train_items), "val_n": len(val_items), "test_n": len(test_items),
    "test_metrics": test_metrics,
    "preprocessing": {"parse_char_budget": PARSE_CHAR_BUDGET,
                      "strip_tags": STRIP_TAGS, "strip_comments": True,
                      "max_tokens": MAX_LEN, "input_format": "cleaned HTML markup as text"},
    "inference": {"operating_threshold": 0.5, "temperature": TEMPERATURE},
    "question": QUESTION,
    "baselines": {"length_only_auc_test": length_baseline,
                  "markuplm_v1_auc": 0.869, "laya_zero_shot_auc_mendeley": 0.724},
    "objective": "cross-entropy, matching train_html_phishing_classifier.ipynb; "
                 "not Laya's RLCD reinforcement objective",
    "note": "Same 756-page sample as MarkupLM v1, so the two are directly comparable.",
}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
print("saved to", ARTIFACT_DIR.resolve())
for item in sorted(ARTIFACT_DIR.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(ARTIFACT_DIR)}  {item.stat().st_size/1e6:.1f} MB")